In [ ]:
# config bootstrap (auto-added): resolve repo paths from config.py
import os as _os, sys as _sys
_h = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_h, 'config.py')) and _os.path.dirname(_h) != _h:
    _h = _os.path.dirname(_h)
_sys.path.insert(0, _h)
import config as _cfg

# Fusion Classifier — Fine-Tuned BLIP ITM + DeBERTa NLI

Loads the best fine-tuned BLIP ITM checkpoint (epoch 9, F1: 75.1%) and combines with DeBERTa NLI entailment scores.
- 1000 samples (500 real + 500 fake)
- 5-fold cross-validation with Logistic Regression
- Compares against previous non-fine-tuned fusion baseline (76% accuracy)

In [1]:
import json
import os
import re

import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm import tqdm
from transformers import (
    BlipProcessor, BlipForImageTextRetrieval,
    AutoTokenizer, AutoModelForSequenceClassification,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    classification_report, confusion_matrix,
)
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

d:\Pics Can Lie\venv\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Device: cuda
GPU: NVIDIA GeForce RTX 4060 Laptop GPU


In [2]:
# ── Config ──
DATASET_ROOT     = _os.path.join(str(_cfg.ROOT), 'datasets', 'dataset')
ANNOTATIONS_PATH = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "merged_balanced", "train.json")
METADATA_PATH    = os.path.join(DATASET_ROOT, "data", "NewsClipPings", "metadata", "train.json")
IMAGE_BASE       = os.path.join(DATASET_ROOT, "origin")
ARTICLE_BASE     = os.path.join(DATASET_ROOT, "origin")

FINETUNED_BLIP_DIR = _os.path.join(str(_cfg.ROOT), 'models', 'blip_itm_finetuned')
NUM_PER_CLASS = 500  # 500 real + 500 fake = 1000 total

def resolve_image_path(meta_image_path: str) -> str:
    rel = meta_image_path.replace("visual_news/", "", 1)
    return os.path.join(IMAGE_BASE, rel)

def resolve_article_path(meta_article_path: str) -> str:
    rel = meta_article_path.replace("visual_news/", "", 1)
    return os.path.join(ARTICLE_BASE, rel)

print("Config ready.")

Config ready.


In [3]:
# ── Load 1000 samples (500 real + 500 fake) ──
with open(ANNOTATIONS_PATH, "r", encoding="utf-8") as f:
    annotations = json.load(f)["annotations"]
with open(METADATA_PATH, "r", encoding="utf-8") as f:
    metadata = json.load(f)

real_samples, fake_samples = [], []

for ann in annotations:
    if len(real_samples) >= NUM_PER_CLASS and len(fake_samples) >= NUM_PER_CLASS:
        break

    art_id = str(ann["id"])
    img_id = str(ann["image_id"])
    if art_id not in metadata or img_id not in metadata:
        continue

    img_path = resolve_image_path(metadata[img_id]["image_path"])
    art_path = resolve_article_path(metadata[img_id]["article_path"])
    if not os.path.isfile(img_path) or not os.path.isfile(art_path):
        continue

    with open(art_path, "r", encoding="utf-8", errors="replace") as f:
        article_text = f.read(2000)
    if len(article_text.strip()) < 50:
        continue

    entry = {
        "article_id": ann["id"],
        "image_id": ann["image_id"],
        "caption": metadata[art_id]["caption"],
        "image_path": img_path,
        "article_text": article_text,
        "falsified": ann["falsified"],
    }

    if not ann["falsified"] and len(real_samples) < NUM_PER_CLASS:
        real_samples.append(entry)
    elif ann["falsified"] and len(fake_samples) < NUM_PER_CLASS:
        fake_samples.append(entry)

samples = real_samples + fake_samples
print(f"Loaded {len(real_samples)} real + {len(fake_samples)} fake = {len(samples)} samples")

Loaded 500 real + 500 fake = 1000 samples


In [4]:
# ── Module 1: Fine-Tuned BLIP ITM scores ──
print(f"Loading fine-tuned BLIP from {FINETUNED_BLIP_DIR}...")
blip_processor = BlipProcessor.from_pretrained(FINETUNED_BLIP_DIR)
blip_model = BlipForImageTextRetrieval.from_pretrained(FINETUNED_BLIP_DIR).to(device).eval()

trainable = sum(p.numel() for p in blip_model.parameters()) / 1e6
print(f"Model loaded on {device} ({trainable:.1f}M params)")

itm_scores = []
for s in tqdm(samples, desc="BLIP ITM (fine-tuned)"):
    image = Image.open(s["image_path"]).convert("RGB")
    inputs = blip_processor(images=image, text=s["caption"], return_tensors="pt").to(device)
    with torch.no_grad():
        outputs = blip_model(**inputs, use_itm_head=True)
        probs = torch.softmax(outputs.itm_score, dim=1)
        itm_scores.append(probs[0, 1].item())  # P(match)

del blip_model, blip_processor
torch.cuda.empty_cache()
print(f"Done. Scores: min={min(itm_scores):.4f}, max={max(itm_scores):.4f}, mean={np.mean(itm_scores):.4f}")

Loading fine-tuned BLIP from D:\Pics Can Lie\blip_itm_finetuned...


Loading weights: 100%|██████████| 472/472 [00:00<00:00, 3027.99it/s]


Model loaded on cuda (223.7M params)


BLIP ITM (fine-tuned): 100%|██████████| 1000/1000 [01:06<00:00, 15.01it/s]

Done. Scores: min=0.0005, max=0.9995, mean=0.4925


In [5]:
# ── Module 2: DeBERTa NLI entailment scores ──
def extract_top_sentences(article: str, caption: str, top_k: int = 3) -> str:
    sentences = [s.strip() for s in re.split(r'[.!?]+', article) if len(s.strip()) > 20]
    if len(sentences) <= top_k:
        return article
    vectorizer = TfidfVectorizer(stop_words="english")
    tfidf = vectorizer.fit_transform([caption] + sentences)
    sims = cosine_similarity(tfidf[0:1], tfidf[1:])[0]
    top_idx = sims.argsort()[-top_k:][::-1]
    return ". ".join(sentences[i] for i in sorted(top_idx)) + "."

print("Loading DeBERTa NLI model...")
nli_tokenizer = AutoTokenizer.from_pretrained("cross-encoder/nli-deberta-v3-large")
nli_model = AutoModelForSequenceClassification.from_pretrained("cross-encoder/nli-deberta-v3-large").to(device)
nli_model.eval()
print(f"DeBERTa loaded on {device}")

entailment_scores = []
for s in tqdm(samples, desc="DeBERTa NLI"):
    premise = extract_top_sentences(s["article_text"], s["caption"])
    inputs = nli_tokenizer(
        premise, s["caption"],
        return_tensors="pt", truncation=True, max_length=512, padding=True
    ).to(device)
    with torch.no_grad():
        logits = nli_model(**inputs).logits
        probs = torch.softmax(logits, dim=1)[0]
        entailment_scores.append(probs[1].item())  # index 1 = entailment

del nli_model, nli_tokenizer
torch.cuda.empty_cache()
print(f"Done. Scores: min={min(entailment_scores):.4f}, max={max(entailment_scores):.4f}, mean={np.mean(entailment_scores):.4f}")

Loading DeBERTa NLI model...


Loading weights: 100%|██████████| 394/394 [00:00<00:00, 2985.49it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: cross-encoder/nli-deberta-v3-large
Key                             | Status     |  | 
--------------------------------+------------+--+-
deberta.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


DeBERTa loaded on cuda


DeBERTa NLI: 100%|██████████| 1000/1000 [01:24<00:00, 11.90it/s]

Done. Scores: min=0.0000, max=0.9992, mean=0.1620


In [6]:
# ── Build feature matrix ──
labels = [1 if s["falsified"] else 0 for s in samples]

df = pd.DataFrame({
    "article_id": [s["article_id"] for s in samples],
    "image_id": [s["image_id"] for s in samples],
    "itm_score": itm_scores,
    "entailment_score": entailment_scores,
    "label": labels,
    "label_str": ["FAKE" if l else "REAL" for l in labels],
})

print("Feature summary:")
print(df.groupby("label_str")[["itm_score", "entailment_score"]].agg(["mean", "std"]).round(4))
print(f"\nTotal: {len(df)} ({(df['label']==0).sum()} real, {(df['label']==1).sum()} fake)")

df.to_csv(_os.path.join(str(_cfg.ROOT), 'features', 'fusion_features_finetuned_1000.csv'), index=False)
print("Saved to fusion_features_finetuned_1000.csv")

Feature summary:
          itm_score         entailment_score        
               mean     std             mean     std
label_str                                           
FAKE         0.2531  0.2586           0.0752  0.1992
REAL         0.7320  0.2513           0.2487  0.4114

Total: 1000 (500 real, 500 fake)
Saved to fusion_features_finetuned_1000.csv


In [7]:
# ── 5-Fold Cross-Validation ──
X = df[["itm_score", "entailment_score"]].values
y = df["label"].values

print(f"Features: Fine-Tuned BLIP ITM + DeBERTa Entailment")
print(f"Samples: {len(X)} ({sum(y==0)} real, {sum(y==1)} fake)")
print(f"Method: 5-fold stratified cross-validation\n")

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

fold_results = []
all_y_true = []
all_y_pred = []
all_y_prob = []

for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
    X_train, X_test = X[train_idx], X[test_idx]
    y_train, y_test = y[train_idx], y[test_idx]

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    clf = LogisticRegression(random_state=42, max_iter=1000)
    clf.fit(X_train_scaled, y_train)

    y_pred = clf.predict(X_test_scaled)
    y_prob = clf.predict_proba(X_test_scaled)[:, 1]

    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec  = recall_score(y_test, y_pred)
    f1   = f1_score(y_test, y_pred)

    fold_results.append({"fold": fold+1, "accuracy": acc, "precision": prec, "recall": rec, "f1": f1})
    all_y_true.extend(y_test)
    all_y_pred.extend(y_pred)
    all_y_prob.extend(y_prob)

    print(f"  Fold {fold+1}: Acc={acc:.1%}  Prec={prec:.1%}  Rec={rec:.1%}  F1={f1:.1%}")

fold_df = pd.DataFrame(fold_results)
print(f"\n{'=' * 65}")
print(f"5-FOLD CV — FINE-TUNED BLIP ITM + DeBERTa NLI — {len(X)} samples")
print(f"{'=' * 65}")
print(f"  Mean Accuracy  : {fold_df['accuracy'].mean():.2%} (+/- {fold_df['accuracy'].std():.2%})")
print(f"  Mean Precision : {fold_df['precision'].mean():.2%} (+/- {fold_df['precision'].std():.2%})")
print(f"  Mean Recall    : {fold_df['recall'].mean():.2%} (+/- {fold_df['recall'].std():.2%})")
print(f"  Mean F1 Score  : {fold_df['f1'].mean():.2%} (+/- {fold_df['f1'].std():.2%})")
print(f"{'=' * 65}")

all_y_true = np.array(all_y_true)
all_y_pred = np.array(all_y_pred)
all_y_prob = np.array(all_y_prob)

print(f"\nAggregated classification report:")
print(classification_report(all_y_true, all_y_pred, target_names=["REAL", "FAKE"]))

cm = confusion_matrix(all_y_true, all_y_pred)
cm_df = pd.DataFrame(cm, index=["Actual REAL", "Actual FAKE"], columns=["Pred REAL", "Pred FAKE"])
display(cm_df)

print(f"\nFeature weights (last fold):")
for name, coef in zip(["BLIP ITM (fine-tuned)", "DeBERTa Entailment"], clf.coef_[0]):
    print(f"  {name:>25s}: {coef:+.4f}")

Features: Fine-Tuned BLIP ITM + DeBERTa Entailment
Samples: 1000 (500 real, 500 fake)
Method: 5-fold stratified cross-validation

  Fold 1: Acc=81.0%  Prec=81.6%  Rec=80.0%  F1=80.8%
  Fold 2: Acc=80.5%  Prec=80.2%  Rec=81.0%  F1=80.6%
  Fold 3: Acc=81.5%  Prec=82.5%  Rec=80.0%  F1=81.2%
  Fold 4: Acc=85.0%  Prec=84.3%  Rec=86.0%  F1=85.1%
  Fold 5: Acc=87.5%  Prec=86.4%  Rec=89.0%  F1=87.7%

5-FOLD CV — FINE-TUNED BLIP ITM + DeBERTa NLI — 1000 samples
  Mean Accuracy  : 83.10% (+/- 3.03%)
  Mean Precision : 83.01% (+/- 2.42%)
  Mean Recall    : 83.20% (+/- 4.09%)
  Mean F1 Score  : 83.09% (+/- 3.17%)

Aggregated classification report:
              precision    recall  f1-score   support

        REAL       0.83      0.83      0.83       500
        FAKE       0.83      0.83      0.83       500

    accuracy                           0.83      1000
   macro avg       0.83      0.83      0.83      1000
weighted avg       0.83      0.83      0.83      1000



,Pred REAL,Pred FAKE
Actual REAL,415,85
Actual FAKE,84,416



Feature weights (last fold):
      BLIP ITM (fine-tuned): -1.9071
         DeBERTa Entailment: -0.5975


In [8]:
# ── Comparison: Fine-tuned vs Non-fine-tuned baseline ──
ft_acc = fold_df['accuracy'].mean()
ft_f1  = fold_df['f1'].mean()
baseline_acc = 0.76
baseline_f1  = 0.7601

print("=" * 65)
print("COMPARISON: Fine-Tuned vs Non-Fine-Tuned Fusion")
print("=" * 65)
print(f"{'Metric':<20s} {'Baseline (pretrained)':>22s} {'Fine-tuned':>15s} {'Delta':>10s}")
print("-" * 65)
print(f"{'Accuracy':<20s} {baseline_acc:>22.2%} {ft_acc:>15.2%} {ft_acc - baseline_acc:>+10.2%}")
print(f"{'F1 Score':<20s} {baseline_f1:>22.2%} {ft_f1:>15.2%} {ft_f1 - baseline_f1:>+10.2%}")
print("-" * 65)
print(f"\nNote: Baseline was 2000 samples (1000+1000), this run uses 1000 samples (500+500).")
print(f"Both use BLIP ITM + DeBERTa NLI with 5-fold CV Logistic Regression.")

COMPARISON: Fine-Tuned vs Non-Fine-Tuned Fusion
Metric                Baseline (pretrained)      Fine-tuned      Delta
-----------------------------------------------------------------
Accuracy                             76.00%          83.10%     +7.10%
F1 Score                             76.01%          83.09%     +7.08%
-----------------------------------------------------------------

Note: Baseline was 2000 samples (1000+1000), this run uses 1000 samples (500+500).
Both use BLIP ITM + DeBERTa NLI with 5-fold CV Logistic Regression.


In [ ]:
# ── Visualizations ──
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Confusion matrix
ax = axes[0]
im = ax.imshow(cm, cmap="Blues", aspect="auto")
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(["Pred REAL", "Pred FAKE"])
ax.set_yticklabels(["Actual REAL", "Actual FAKE"])
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=18, fontweight="bold")
ax.set_title("Confusion Matrix (All Folds)")

# 2. Per-fold metrics
ax = axes[1]
x = np.arange(5)
w = 0.2
ax.bar(x - 1.5*w, fold_df["accuracy"], w, label="Accuracy", color="#3498db", edgecolor="white")
ax.bar(x - 0.5*w, fold_df["precision"], w, label="Precision", color="#2ecc71", edgecolor="white")
ax.bar(x + 0.5*w, fold_df["recall"], w, label="Recall", color="#e67e22", edgecolor="white")
ax.bar(x + 1.5*w, fold_df["f1"], w, label="F1", color="#e74c3c", edgecolor="white")
ax.axhline(y=baseline_acc, color="gray", linestyle="--", linewidth=1.5, label=f"Baseline ({baseline_acc:.0%})")
ax.set_xticks(x)
ax.set_xticklabels([f"Fold {i+1}" for i in range(5)])
ax.set_ylim(0, 1.05)
ax.set_ylabel("Score")
ax.set_title("Per-Fold Metrics")
ax.legend(fontsize=7)

# 3. Fusion probability distribution
ax = axes[2]
real_probs = all_y_prob[all_y_true == 0]
fake_probs = all_y_prob[all_y_true == 1]
ax.hist(real_probs, bins=15, alpha=0.7, color="#2ecc71", label="REAL", edgecolor="white")
ax.hist(fake_probs, bins=15, alpha=0.7, color="#e74c3c", label="FAKE", edgecolor="white")
ax.axvline(x=0.5, color="orange", linestyle="--", linewidth=2, label="Threshold")
ax.set_xlabel("Predicted FAKE Probability")
ax.set_ylabel("Count")
ax.set_title("Classifier Output (All Folds)")
ax.legend()

plt.suptitle(f"Fusion (Fine-Tuned BLIP + DeBERTa) — Acc: {ft_acc:.1%} | F1: {ft_f1:.1%} (baseline: {baseline_acc:.0%})",
             fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig(_os.path.join(str(_cfg.ROOT), 'results', 'fusion_finetuned_results.png'), dpi=150, bbox_inches="tight")
plt.show()
print("Plot saved to fusion_finetuned_results.png")